# Fashion-MNIST: Autoencoder + Classifier

Complete Track C capstone notebook. Run the cells from top to bottom. Training time depends on your computer; a GPU is recommended.

## 1. Imports and settings

In [ ]:
# Import the libraries and choose CPU or GPU.
import os, random, copy, math, warnings, time
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import datasets, transforms

from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, ConfusionMatrixDisplay,
                             silhouette_score, roc_auc_score, classification_report)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 256
LATENT_DIMS = [16, 32, 64, 128]
NUM_CLASSES = 10
# Increase these for stronger final experiments.
EPOCHS_AE = 2
EPOCHS_CLASSIFIER = 3
EPOCHS_ADVANCED = 2
LEARNING_RATE = 1e-3
NUM_WORKERS = 2 if os.name != "nt" else 0
print("Device:", DEVICE)

## 2. Load Fashion-MNIST

In [ ]:
# Download the dataset. Images are converted to tensors with values from 0 to 1.
transform = transforms.ToTensor()
train_data = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

CLASS_NAMES = train_data.classes
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
    x, y = train_data[i]
    ax.imshow(x.squeeze(), cmap="gray"); ax.set_title(CLASS_NAMES[y], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Define the autoencoders

In [ ]:
# Each model compresses an image into a small vector, then rebuilds the image.
class VanillaAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(),
                                     nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 256), nn.ReLU(),
                                     nn.Linear(256, 512), nn.ReLU(), nn.Linear(512, 784),
                                     nn.Sigmoid())
    def encode(self, x): return self.encoder(x)
    def decode(self, z): return self.decoder(z).view(-1, 1, 28, 28)
    def forward(self, x): return self.decode(self.encode(x))

class ConvAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv2d(1, 32, 3, 2, 1), nn.ReLU(),
                                  nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
                                  nn.Flatten(), nn.Linear(64*7*7, latent_dim))
        self.fc = nn.Sequential(nn.Linear(latent_dim, 64*7*7), nn.ReLU())
        self.deconv = nn.Sequential(nn.Unflatten(1, (64, 7, 7)),
                                    nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
                                    nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid())
    def encode(self, x): return self.conv(x)
    def decode(self, z): return self.deconv(self.fc(z))
    def forward(self, x): return self.decode(self.encode(x))

class DenoisingAE(ConvAE):
    pass

class VAE(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 32, 3, 2, 1), nn.ReLU(),
                                      nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(), nn.Flatten())
        self.mu = nn.Linear(64*7*7, latent_dim)
        self.logvar = nn.Linear(64*7*7, latent_dim)
        self.decoder_fc = nn.Sequential(nn.Linear(latent_dim, 64*7*7), nn.ReLU())
        self.decoder = nn.Sequential(nn.Unflatten(1, (64, 7, 7)),
                                     nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(),
                                     nn.ConvTranspose2d(32, 1, 4, 2, 1), nn.Sigmoid())
    def encode_stats(self, x):
        h = self.features(x); return self.mu(h), self.logvar(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def encode(self, x): return self.encode_stats(x)[0]
    def decode(self, z): return self.decoder(self.decoder_fc(z))
    def forward(self, x):
        mu, logvar = self.encode_stats(x); z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

class ContractiveAE(ConvAE):
    # Contractive penalty is computed on the encoder's latent output.
    pass

def make_autoencoder(kind, latent_dim):
    if kind == "vanilla": return VanillaAE(latent_dim)
    if kind in ("conv", "denoising", "contractive"): return {"conv": ConvAE, "denoising": DenoisingAE,
                                                               "contractive": ContractiveAE}[kind](latent_dim)
    if kind == "vae": return VAE(latent_dim)
    raise ValueError(kind)

def contractive_penalty(model, x, z):
    # Hutchinson-style approximation of ||J_encoder||_F^2 using one random projection.
    v = torch.randn_like(z)
    scalar = (z * v).sum()
    grad = torch.autograd.grad(scalar, x, create_graph=True, retain_graph=True)[0]
    return grad.pow(2).flatten(1).sum(1).mean()

## 4. Train and evaluate autoencoders

In [ ]:
# Train without labels. Each model learns to reconstruct its input.
def add_noise(x, sigma=0.35):
    return (x + sigma * torch.randn_like(x)).clamp(0, 1)

def train_autoencoder(model, kind, loader=train_loader, epochs=EPOCHS_AE,
                      contractive_lambda=1e-4, beta=1.0):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history = []
    for epoch in range(epochs):
        model.train(); total = 0.0
        for x, _ in loader:
            x = x.to(DEVICE)
            opt.zero_grad()
            if kind == "vae":
                recon, mu, logvar = model(x)
                rec = F.binary_cross_entropy(recon, x, reduction="sum") / x.size(0)
                kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
                loss = rec + beta * kl
            elif kind == "denoising":
                recon = model(add_noise(x))
                loss = F.mse_loss(recon, x)
            elif kind == "contractive":
                x_req = x.detach().requires_grad_(True)
                z = model.encode(x_req); recon = model.decode(z)
                loss = F.mse_loss(recon, x_req) + contractive_lambda * contractive_penalty(model, x_req, z)
            else:
                recon = model(x); loss = F.mse_loss(recon, x)
            loss.backward(); opt.step()
            total += loss.item() * x.size(0)
        history.append(total / len(loader.dataset))
        print(f"epoch {epoch+1}/{epochs} - loss {history[-1]:.5f}")
    return model, history

@torch.no_grad()
def reconstruction_metrics(model, loader=test_loader, kind="conv"):
    model.eval(); mse_sum=mae_sum=n=0
    for x, _ in loader:
        x=x.to(DEVICE)
        out=model(x)
        recon=out[0] if kind=="vae" else out
        mse_sum += F.mse_loss(recon, x, reduction="sum").item()
        mae_sum += F.l1_loss(recon, x, reduction="sum").item()
        n += x.numel()
    return {"MSE": mse_sum/n, "MAE": mae_sum/n}

@torch.no_grad()
def show_reconstructions(model, kind, loader=test_loader, count=8):
    model.eval(); x,_=next(iter(loader)); x=x[:count].to(DEVICE)
    out=model(x); recon=out[0] if kind=="vae" else out
    fig,ax=plt.subplots(2,count,figsize=(count*1.5,3))
    for i in range(count):
        ax[0,i].imshow(x[i,0].cpu(),cmap="gray"); ax[0,i].axis("off")
        ax[1,i].imshow(recon[i,0].cpu(),cmap="gray"); ax[1,i].axis("off")
    ax[0,0].set_ylabel("Original"); ax[1,0].set_ylabel("Reconstruction")
    plt.tight_layout(); plt.show()

## 5. Train autoencoders

The notebook starts in quick mode (ConvAE with 16 features) so it is easier to test. Set `RUN_FULL_AE_GRID = True` in the next code cell to run all five autoencoders with latent sizes 16, 32, 64, and 128.

In [ ]:
# Train the requested model types using each selected latent size.
AE_KINDS = ["vanilla", "conv", "denoising", "vae", "contractive"]
ae_models, ae_histories, ae_rows = {}, {}, []
RUN_FULL_AE_GRID = False  # Change to True to train all 5 types with all 4 latent sizes
kinds_to_run = AE_KINDS if RUN_FULL_AE_GRID else ["conv"]
dims_to_run = LATENT_DIMS if RUN_FULL_AE_GRID else [16]

for kind in kinds_to_run:
    for dim in dims_to_run:
        print(f"\nTraining {kind} AE, latent dimension={dim}")
        model, hist = train_autoencoder(make_autoencoder(kind, dim), kind)
        ae_models[(kind,dim)] = model
        ae_histories[(kind,dim)] = hist
        metrics = reconstruction_metrics(model, kind=kind)
        ae_rows.append({"Autoencoder":kind, "Latent dim":dim, **metrics})
ae_results = pd.DataFrame(ae_rows)
display(ae_results.sort_values("MSE"))

## 6. View reconstructions and extract features

In [ ]:
# Compare original images with reconstructions and collect encoded features.
# Show an example reconstruction from the 16-dimensional convolutional AE.
if ("conv",16) in ae_models:
    show_reconstructions(ae_models[("conv",16)], "conv")

@torch.no_grad()
def extract_features(model, loader, kind):
    model.eval(); feats=[]; labels=[]
    for x,y in loader:
        x=x.to(DEVICE)
        if kind=="vae":
            z=model.encode(x)
        else:
            z=model.encode(x)
        feats.append(z.cpu()); labels.append(y)
    return torch.cat(feats).numpy(), torch.cat(labels).numpy()

feature_cache={}
for key, model in ae_models.items():
    kind,dim=key
    feature_cache[key] = {
        "train": extract_features(model, train_loader, kind),
        "test": extract_features(model, test_loader, kind)
    }
print("Feature sets extracted:", len(feature_cache))

## 7. Visualize latent features

In [ ]:
# Plot the encoded features in two dimensions.
def plot_latent_space(features, labels, title, max_points=3000):
    rng=np.random.default_rng(SEED)
    idx=rng.choice(len(features), min(max_points,len(features)), replace=False)
    X=features[idx]; y=labels[idx]
    pca=PCA(n_components=2, random_state=SEED).fit_transform(X)
    tsne=TSNE(n_components=2, init="pca", learning_rate="auto",
              random_state=SEED, perplexity=30).fit_transform(X)
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    for ax,emb,name in zip(axes,[pca,tsne],["PCA","t-SNE"]):
        sc=ax.scatter(emb[:,0],emb[:,1],c=y,s=5,cmap="tab10",alpha=.65)
        ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title); plt.tight_layout(); plt.show()

if ("conv",16) in feature_cache:
    Xtr,ytr=feature_cache[("conv",16)]["train"]
    plot_latent_space(Xtr,ytr,"ConvAE latent space (train subset)")
try:
    import umap
    if ("conv",16) in feature_cache:
        Xtr,ytr=feature_cache[("conv",16)]["train"]
        emb=umap.UMAP(random_state=SEED).fit_transform(Xtr[:5000])
        plt.figure(figsize=(6,5)); plt.scatter(emb[:,0],emb[:,1],c=ytr[:5000],s=5,cmap="tab10")
        plt.title("UMAP latent space"); plt.show()
except ImportError:
    print("UMAP not installed; t-SNE and PCA are provided. Install with: pip install umap-learn")

## 8. Define classifiers and evaluation functions

In [ ]:
# Define simple raw-image classifiers and common evaluation functions.
class RawMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Flatten(),nn.Linear(784,256),nn.ReLU(),nn.Dropout(.3),
                               nn.Linear(256,128),nn.ReLU(),nn.Linear(128,10))
    def forward(self,x): return self.net(x)

class RawCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                               nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                               nn.Flatten(),nn.Linear(64*7*7,256),nn.ReLU(),nn.Dropout(.3),
                               nn.Linear(256,10))
    def forward(self,x): return self.net(x)

class LatentClassifier(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(latent_dim,256),nn.ReLU(),nn.Dropout(.3),
                               nn.Linear(256,NUM_CLASSES))
    def forward(self,z): return self.net(z)

def fit_classifier(model, train_loader, epochs=EPOCHS_CLASSIFIER, input_mode="image"):
    model=model.to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE)
    for ep in range(epochs):
        model.train(); running=0
        for x,y in train_loader:
            x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad()
            logits=model(x); loss=F.cross_entropy(logits,y)
            loss.backward(); opt.step(); running+=loss.item()*len(y)
        print(f"epoch {ep+1}/{epochs} loss={running/len(train_loader.dataset):.4f}")
    return model

@torch.no_grad()
def evaluate_classifier(model, loader, feature_model=None, feature_kind=None):
    model.eval(); ys=[]; ps=[]
    for x,y in loader:
        x=x.to(DEVICE)
        if feature_model is not None:
            feature_model.eval()
            z=feature_model.encode(x)
            logits=model(z)
        else: logits=model(x)
        ps.extend(logits.argmax(1).cpu().numpy()); ys.extend(y.numpy())
    p,r,f,_=precision_recall_fscore_support(ys,ps,average="macro",zero_division=0)
    return {"accuracy":accuracy_score(ys,ps),"macro_precision":p,"macro_recall":r,"macro_f1":f,
            "labels":np.array(ys),"predictions":np.array(ps)}

def plot_confusion(result,title):
    ConfusionMatrixDisplay(confusion_matrix(result["labels"],result["predictions"]),
                           display_labels=CLASS_NAMES).plot(xticks_rotation=45,cmap="Blues")
    plt.title(title); plt.tight_layout(); plt.show()

## 9. Approach A: frozen encoder

In [ ]:
# Train a classifier using features from a fixed, pre-trained encoder.
def latent_loaders(model, kind, batch_size=BATCH_SIZE):
    Xtr,ytr=extract_features(model,train_loader,kind)
    Xte,yte=extract_features(model,test_loader,kind)
    tr=DataLoader(TensorDataset(torch.tensor(Xtr,dtype=torch.float32),
                                torch.tensor(ytr,dtype=torch.long)),batch_size=batch_size,shuffle=True)
    te=DataLoader(TensorDataset(torch.tensor(Xte,dtype=torch.float32),
                                torch.tensor(yte,dtype=torch.long)),batch_size=batch_size)
    return tr,te

classification_rows=[]
# Raw MLP and CNN baselines
raw_mlp=fit_classifier(RawMLP(),train_loader)
raw_mlp_result=evaluate_classifier(raw_mlp,test_loader)
classification_rows.append({"Method":"Raw MLP","AE":"-","latent_dim":"-",**{k:v for k,v in raw_mlp_result.items() if isinstance(v,float)}})
raw_cnn=fit_classifier(RawCNN(),train_loader)
raw_cnn_result=evaluate_classifier(raw_cnn,test_loader)
classification_rows.append({"Method":"Raw CNN","AE":"-","latent_dim":"-",**{k:v for k,v in raw_cnn_result.items() if isinstance(v,float)}})

frozen_classifiers={}
for (kind,dim), ae in ae_models.items():
    tr_loader,te_loader=latent_loaders(ae,kind)
    clf=fit_classifier(LatentClassifier(dim),tr_loader)
    res=evaluate_classifier(clf,te_loader)
    frozen_classifiers[(kind,dim)]=clf
    classification_rows.append({"Method":"Frozen encoder (A)","AE":kind,"latent_dim":dim,
                                **{k:v for k,v in res.items() if isinstance(v,float)}})
classification_results=pd.DataFrame(classification_rows)
display(classification_results.sort_values("accuracy",ascending=False))

## 10. Approach B: fine-tune the encoder

In [ ]:
# Update the encoder and classifier together using class labels.
class EndToEndClassifier(nn.Module):
    def __init__(self, encoder_model, latent_dim, kind):
        super().__init__(); self.encoder_model=encoder_model; self.kind=kind
        self.head=LatentClassifier(latent_dim)
    def forward(self,x):
        z=self.encoder_model.encode(x)
        return self.head(z)

def train_end_to_end(ae, kind, latent_dim, epochs=EPOCHS_CLASSIFIER):
    model=EndToEndClassifier(ae,latent_dim,kind).to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE*0.5)
    for ep in range(epochs):
        model.train(); total=0
        for x,y in train_loader:
            x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad()
            loss=F.cross_entropy(model(x),y); loss.backward(); opt.step(); total+=loss.item()*len(y)
        print(f"fine-tune epoch {ep+1}/{epochs}: {total/len(train_loader.dataset):.4f}")
    return model

end_to_end_models={}
for (kind,dim), ae in ae_models.items():
    tuned= train_end_to_end(copy.deepcopy(ae),kind,dim)
    end_to_end_models[(kind,dim)]=tuned
    res=evaluate_classifier(tuned,test_loader)
    classification_rows.append({"Method":"Fine-tuned encoder (B)","AE":kind,"latent_dim":dim,
                                **{k:v for k,v in res.items() if isinstance(v,float)}})

## 11. Approach C: joint reconstruction and classification

In [ ]:
# Train the encoder, decoder, and classifier together.
class JointAEClassifier(nn.Module):
    def __init__(self, ae, latent_dim):
        super().__init__(); self.ae=ae; self.classifier=LatentClassifier(latent_dim)
    def forward(self,x):
        z=self.ae.encode(x); recon=self.ae.decode(z); logits=self.classifier(z)
        return recon,logits

def train_joint(ae,kind,latent_dim,alpha=1.0,beta=1.0,epochs=EPOCHS_ADVANCED):
    model=JointAEClassifier(copy.deepcopy(ae),latent_dim).to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=LEARNING_RATE*0.5)
    for ep in range(epochs):
        model.train()
        for x,y in train_loader:
            x=x.to(DEVICE); y=y.to(DEVICE); opt.zero_grad()
            recon,logits=model(x)
            rec=F.mse_loss(recon,x); cls=F.cross_entropy(logits,y)
            loss=alpha*rec+beta*cls
            loss.backward(); opt.step()
        print(f"joint epoch {ep+1}: rec={rec.item():.4f}, cls={cls.item():.4f}")
    return model

joint_models={}
if ("conv",16) in ae_models:
    for alpha,beta in [(0.1,1.0),(1.0,1.0),(1.0,0.1)]:
        jm=train_joint(ae_models[("conv",16)],"conv",16,alpha,beta)
        joint_models[(alpha,beta)]=jm
        res=evaluate_classifier(jm,test_loader)
        classification_rows.append({"Method":f"Joint C (a={alpha},b={beta})","AE":"conv","latent_dim":16,
                                    **{k:v for k,v in res.items() if isinstance(v,float)}})
classification_results=pd.DataFrame(classification_rows)
display(classification_results.sort_values("accuracy",ascending=False))

## 12. Semi-supervised learning (10% labeled data)

In [ ]:
# Compare training with 10% labels against a model trained from scratch.
seed_everything()
n_labeled=max(1,int(0.10*len(train_data)))
labeled_indices=np.random.choice(len(train_data),n_labeled,replace=False).tolist()
labeled_subset=Subset(train_data,labeled_indices)
labeled_loader=DataLoader(labeled_subset,batch_size=BATCH_SIZE,shuffle=True)
print(f"Labeled examples: {n_labeled}/{len(train_data)} ({100*n_labeled/len(train_data):.1f}%)")

# Supervised-only baseline on the 10% labeled subset
supervised_10=fit_classifier(RawCNN(),labeled_loader,epochs=EPOCHS_CLASSIFIER)
supervised_10_result=evaluate_classifier(supervised_10,test_loader)

# Pre-trained encoder + classifier, trained using the same labeled subset
semi_ae=ae_models.get(("conv",16))
semi_model=EndToEndClassifier(copy.deepcopy(semi_ae),16,"conv").to(DEVICE)
semi_opt=torch.optim.Adam(semi_model.parameters(),lr=LEARNING_RATE*0.5)
for ep in range(EPOCHS_CLASSIFIER):
    semi_model.train()
    for x,y in labeled_loader:
        x=x.to(DEVICE); y=y.to(DEVICE); semi_opt.zero_grad()
        loss=F.cross_entropy(semi_model(x),y); loss.backward(); semi_opt.step()
    print(f"semi-supervised fine-tune epoch {ep+1}")
semi_result=evaluate_classifier(semi_model,test_loader)
semi_supervised_results=pd.DataFrame([
    {"Method":"Supervised-only (10% labels)",**{k:v for k,v in supervised_10_result.items() if isinstance(v,float)}},
    {"Method":"AE pretrain + 10% labels",**{k:v for k,v in semi_result.items() if isinstance(v,float)}}
])
display(semi_supervised_results)

## 13. Adversarial autoencoder

In [ ]:
# The discriminator checks whether encoded vectors look like samples from a normal distribution.
class AAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.encoder=nn.Sequential(nn.Flatten(),nn.Linear(784,512),nn.ReLU(),nn.Linear(512,latent_dim))
        self.decoder=nn.Sequential(nn.Linear(latent_dim,512),nn.ReLU(),nn.Linear(512,784),nn.Sigmoid())
        self.discriminator=nn.Sequential(nn.Linear(latent_dim,128),nn.LeakyReLU(.2),
                                         nn.Linear(128,1))
    def encode(self,x): return self.encoder(x)
    def decode(self,z): return self.decoder(z).view(-1,1,28,28)
    def forward(self,x): return self.decode(self.encode(x))

def train_aae(latent_dim=32,epochs=EPOCHS_ADVANCED):
    model=AAE(latent_dim).to(DEVICE)
    opt_ae=torch.optim.Adam(list(model.encoder.parameters())+list(model.decoder.parameters()),lr=LEARNING_RATE)
    opt_d=torch.optim.Adam(model.discriminator.parameters(),lr=LEARNING_RATE)
    bce=nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        for x,_ in train_loader:
            x=x.to(DEVICE)
            # Discriminator: encoded samples are fake, Gaussian prior samples are real.
            z_real=torch.randn(x.size(0),latent_dim,device=DEVICE)
            z_fake=model.encode(x).detach()
            opt_d.zero_grad()
            d_loss=bce(model.discriminator(z_real),torch.ones(x.size(0),1,device=DEVICE))+bce(
                model.discriminator(z_fake),torch.zeros(x.size(0),1,device=DEVICE))
            d_loss.backward(); opt_d.step()
            # Encoder/decoder: reconstruction + fool discriminator.
            opt_ae.zero_grad()
            z=model.encode(x); recon=model.decode(z)
            rec=F.mse_loss(recon,x)
            adv=bce(model.discriminator(z),torch.ones(x.size(0),1,device=DEVICE))
            (rec+0.1*adv).backward(); opt_ae.step()
        print(f"AAE epoch {ep+1}: recon={rec.item():.4f}, discriminator={d_loss.item():.4f}")
    return model

aae_model=train_aae(32)
aae_features_test,aae_labels_test=extract_features(aae_model,test_loader,"aae") if False else (None,None)
# AAE encoder can be used with the same latent classifier.
aae_train_X,aae_train_y=extract_features(aae_model,train_loader,"conv") if False else (None,None)

# Classification performance using AAE latent representations.
aae_train_features, aae_train_labels = extract_features(aae_model, train_loader, "aae")
aae_test_features, aae_test_labels = extract_features(aae_model, test_loader, "aae")
aae_train_latent_loader = DataLoader(
    TensorDataset(torch.tensor(aae_train_features,dtype=torch.float32),
                  torch.tensor(aae_train_labels,dtype=torch.long)),
    batch_size=BATCH_SIZE, shuffle=True)
aae_test_latent_loader = DataLoader(
    TensorDataset(torch.tensor(aae_test_features,dtype=torch.float32),
                  torch.tensor(aae_test_labels,dtype=torch.long)),
    batch_size=BATCH_SIZE)
aae_classifier = fit_classifier(LatentClassifier(32), aae_train_latent_loader)
aae_result = evaluate_classifier(aae_classifier, aae_test_latent_loader)
classification_rows.append({"Method":"AAE latent classifier","AE":"AAE","latent_dim":32,
                            **{k:v for k,v in aae_result.items() if isinstance(v,float)}})
print("AAE latent classifier:", {k:v for k,v in aae_result.items() if isinstance(v,float)})


## 14. Anomaly detection

In [ ]:
# Use reconstruction error as an anomaly score.
@torch.no_grad()
def reconstruction_errors(model, loader):
    model.eval(); errs=[]; labels=[]
    for x,y in loader:
        x=x.to(DEVICE); recon=model(x)
        if isinstance(recon,tuple): recon=recon[0]
        err=(recon-x).pow(2).flatten(1).mean(1)
        errs.extend(err.cpu().numpy()); labels.extend(y.numpy())
    return np.array(errs),np.array(labels)

anomaly_ae=ae_models.get(("conv",16))
normal_train_err,normal_train_labels=reconstruction_errors(anomaly_ae,train_loader)
normal_scores=normal_train_err[normal_train_labels==0]
anomaly_threshold=float(np.quantile(normal_scores,0.95))
test_err,test_labels=reconstruction_errors(anomaly_ae,test_loader)
semantic_anomaly=(test_labels!=0).astype(int)
semantic_pred=(test_err>anomaly_threshold).astype(int)
print("Threshold (95th percentile normal train error):",anomaly_threshold)
print(classification_report(semantic_anomaly,semantic_pred,target_names=["normal class","other classes"],zero_division=0))
try: print("Semantic anomaly ROC-AUC:",roc_auc_score(semantic_anomaly,test_err))
except ValueError: pass

# Corruption-based OOD evaluation: noisy versions of normal-class test images.
@torch.no_grad()
def corrupted_normal_scores(model, dataset, n=2000, sigma=0.45):
    idx=[i for i in range(len(dataset)) if dataset[i][1]==0][:n]
    xs=torch.stack([dataset[i][0] for i in idx]).to(DEVICE)
    noisy=add_noise(xs,sigma)
    recon=model(noisy)
    return (recon-noisy).pow(2).flatten(1).mean(1).cpu().numpy()

corrupt_scores=corrupted_normal_scores(anomaly_ae,test_data)
clean_scores=test_err[test_labels==0]
plt.figure(figsize=(7,4))
plt.hist(clean_scores,bins=50,alpha=.6,label="Clean normal class",density=True)
plt.hist(corrupt_scores,bins=50,alpha=.6,label="Corrupted normal class",density=True)
plt.axvline(anomaly_threshold,linestyle="--",label="Threshold")
plt.xlabel("Reconstruction error"); plt.ylabel("Density"); plt.legend()
plt.title("Anomaly score distribution"); plt.show()

# Quantitative corrupted/OOD evaluation: clean normal samples are negative, corrupted samples positive.
ood_scores = np.concatenate([clean_scores, corrupt_scores])
ood_targets = np.concatenate([np.zeros(len(clean_scores),dtype=int),
                              np.ones(len(corrupt_scores),dtype=int)])
ood_predictions = (ood_scores > anomaly_threshold).astype(int)
ood_p, ood_r, ood_f1, _ = precision_recall_fscore_support(
    ood_targets, ood_predictions, average="binary", zero_division=0)
print({
    "OOD_precision": ood_p,
    "OOD_recall": ood_r,
    "OOD_f1": ood_f1,
    "OOD_ROC_AUC": roc_auc_score(ood_targets, ood_scores)
})


## 15. Feature quality: silhouette score and PCA

In [ ]:
# Compare how well latent features group images by class.
def safe_silhouette(X,y,max_points=4000):
    rng=np.random.default_rng(SEED); idx=rng.choice(len(X),min(max_points,len(X)),replace=False)
    return silhouette_score(X[idx],y[idx])

feature_quality=[]
if ("conv",16) in feature_cache:
    X,y=feature_cache[("conv",16)]["train"]
    feature_quality.append({"Features":"ConvAE latent","Silhouette":safe_silhouette(X,y)})
    # PCA baseline with same dimensionality as the AE code.
    pca_features=PCA(n_components=16,random_state=SEED).fit_transform(
        train_data.data.numpy().reshape(-1,784).astype("float32")/255.0)
    feature_quality.append({"Features":"PCA (16 components)","Silhouette":safe_silhouette(pca_features,y)})
for (kind,dim),cache in feature_cache.items():
    X,y=cache["train"]
    feature_quality.append({"Features":f"{kind} latent ({dim})","Silhouette":safe_silhouette(X,y)})
feature_quality_df=pd.DataFrame(feature_quality).drop_duplicates("Features")
display(feature_quality_df.sort_values("Silhouette",ascending=False))

## 16. Ablation study

In [ ]:
# Summarize the effects of latent size and autoencoder type.
if len(ae_results):
    print("Reconstruction ablation by latent size and AE type")
    display(ae_results.pivot_table(index="Latent dim",columns="Autoencoder",values="MSE"))
if len(classification_results):
    print("Frozen-feature classification ablation")
    display(classification_results[classification_results["Method"]=="Frozen encoder (A)"]
            .pivot_table(index="latent_dim",columns="AE",values="accuracy"))

## 17. Latent-space interpolation

In [ ]:
# Decode points between two encoded images.
@torch.no_grad()
def latent_interpolation(model, dataset=test_data, i=3, j=15, steps=8):
    model.eval()
    x1=dataset[i][0].unsqueeze(0).to(DEVICE); x2=dataset[j][0].unsqueeze(0).to(DEVICE)
    z1=model.encode(x1); z2=model.encode(x2)
    alphas=torch.linspace(0,1,steps,device=DEVICE)
    zs=torch.cat([(1-a)*z1+a*z2 for a in alphas],dim=0)
    imgs=model.decode(zs).cpu()
    fig,axes=plt.subplots(1,steps,figsize=(steps*1.5,2))
    for k,ax in enumerate(axes):
        ax.imshow(imgs[k,0],cmap="gray"); ax.set_title(f"{alphas[k].item():.1f}"); ax.axis("off")
    plt.suptitle("Latent-space interpolation"); plt.show()

if anomaly_ae is not None:
    latent_interpolation(anomaly_ae)

## 18. Feature importance with gradients

In [ ]:
# Use input gradients to highlight pixels that affect the predicted class.
def gradient_saliency(ae, classifier, image, kind="conv"):
    ae.eval(); classifier.eval()
    x=image.unsqueeze(0).to(DEVICE).requires_grad_(True)
    z=ae.encode(x); logits=classifier(z)
    cls=logits.argmax(1).item()
    score=logits[0,cls]
    grad=torch.autograd.grad(score,x)[0].abs()[0,0].detach().cpu().numpy()
    return x.detach().cpu()[0,0].numpy(),grad,cls

if ("conv",16) in ae_models and ("conv",16) in frozen_classifiers:
    sample,label=test_data[0]
    original,saliency,pred=gradient_saliency(ae_models[("conv",16)],
                                             frozen_classifiers[("conv",16)],sample)
    fig,ax=plt.subplots(1,2,figsize=(6,3))
    ax[0].imshow(original,cmap="gray"); ax[0].set_title(f"Input (true: {CLASS_NAMES[label]})")
    ax[1].imshow(saliency,cmap="hot"); ax[1].set_title(f"Gradient saliency (pred: {CLASS_NAMES[pred]})")
    for a in ax:a.axis("off")
    plt.tight_layout(); plt.show()

## 19. Reconstruction error by class

In [ ]:
# Calculate average reconstruction error for each clothing category.
@torch.no_grad()
def reconstruction_error_by_class(model,loader,kind="conv"):
    sums=np.zeros(NUM_CLASSES); counts=np.zeros(NUM_CLASSES)
    model.eval()
    for x,y in loader:
        x=x.to(DEVICE); out=model(x); recon=out[0] if kind=="vae" else out
        per=(recon-x).pow(2).flatten(1).mean(1).cpu().numpy()
        for cls in range(NUM_CLASSES):
            mask=(y.numpy()==cls); sums[cls]+=per[mask].sum(); counts[cls]+=mask.sum()
    return sums/counts

class_errors=reconstruction_error_by_class(anomaly_ae,test_loader)
plt.figure(figsize=(10,4)); plt.bar(CLASS_NAMES,class_errors)
plt.xticks(rotation=45,ha="right"); plt.ylabel("Mean per-image MSE")
plt.title("Reconstruction quality by Fashion-MNIST class"); plt.tight_layout(); plt.show()

## 20. Save results

In [ ]:
# Save result tables and selected model weights.
# Refresh and display all measured classification results.
classification_results=pd.DataFrame(classification_rows)
if len(classification_results):
    display(classification_results.sort_values("accuracy",ascending=False).reset_index(drop=True))
if len(ae_results):
    display(ae_results.sort_values("MSE").reset_index(drop=True))

os.makedirs("capstone_outputs",exist_ok=True)
if len(ae_results): ae_results.to_csv("capstone_outputs/autoencoder_reconstruction_results.csv",index=False)
if len(classification_results): classification_results.to_csv("capstone_outputs/classification_results.csv",index=False)
if len(feature_quality_df): feature_quality_df.to_csv("capstone_outputs/feature_quality_results.csv",index=False)

# Save the principal model weights.
if ("conv",16) in ae_models:
    torch.save(ae_models[("conv",16)].state_dict(),"capstone_outputs/conv_ae_latent16.pt")
if raw_cnn is not None:
    torch.save(raw_cnn.state_dict(),"capstone_outputs/raw_cnn_baseline.pt")
print("Saved outputs in ./capstone_outputs/")

## Final notes

Run the notebook from top to bottom. Quick mode checks that the workflow works. For the complete capstone experiments, switch `RUN_FULL_AE_GRID` to `True` and run again. Use only the results generated by your run.